## Effective use of the Claude Thinking API

### Installing Utilities and Libraries

In [ ]:
%pip install \
    databricks-sdk==0.49.0 \
    anthropic==0.120.2 \
    "mlflow>=3.1"

### Restart your Python Environment

In [ ]:
dbutils.library.restartPython()

### Set up your Environment

In [ ]:
from databricks.sdk import WorkspaceClient

# Get Databricks runtime authentication
w = WorkspaceClient()

headers = w.config.authenticate()
token = headers["Authorization"].replace("Bearer ", "")
workspace_host = w.config.host.rstrip("/")

### Create the Anthropic Client

In [ ]:
import anthropic

# Anthropic client through Databricks
client = anthropic.Anthropic(
    api_key="unused",
    base_url=f"{workspace_host}/serving-endpoints/anthropic",
    default_headers={
        "Authorization": f"Bearer {token}"
    }
)

### Define the User Prompt

In [ ]:
user_prompt = """
You are a senior digital marketing consultant.

A client's e-commerce website has experienced the following issues over the past three months:

- Website traffic has decreased by 35%
- Conversion rate has dropped from 3.8% to 1.7%
- Bounce rate has increased to 72%
- Mobile visitors account for 80% of all traffic

Recommend a recovery strategy.

For each recommendation:

- Explain why it should be prioritized.
- Describe its expected business impact.
- Identify any risks or trade-offs.
"""

### Configure Adaptive Thinking with the Claude API

In [ ]:
response = client.messages.create(
    model = "databricks-claude-sonnet-5",
    max_tokens = 20000,
    thinking = {"type": "adaptive", "display": "summarized"},
    messages = [
        {
            "role": "user",
            "content": user_prompt
        }
    ]
)

for block in response.content:
    if block.type == "thinking":
        print(f"\nThinking: {block.thinking}")
    elif block.type == "text":
        print(f"\nResponse: {block.text}")